# Dice Fairness: Binomial Tests

Die faces: `['oo', 'o', 'xx', 'x', 'x', 'R']`

For each symbol, we run a one-vs-rest binomial test comparing observed count to expected probability from the die design.

In [1]:
import sqlite3
from collections import Counter

import pandas as pd
from scipy.stats import binomtest

In [2]:
DB_PATH = '../dice_rolls.db'
FACES = ['oo', 'o', 'xx', 'x', 'x', 'R']
expected_probs = {k: v / len(FACES) for k, v in Counter(FACES).items()}
expected_probs

{'oo': 0.16666666666666666,
 'o': 0.16666666666666666,
 'xx': 0.16666666666666666,
 'x': 0.3333333333333333,
 'R': 0.16666666666666666}

In [3]:
con = sqlite3.connect(DB_PATH)
rolls = pd.read_sql_query('SELECT result, rolled_at FROM rolls', con)
con.close()

rolls.head(), len(rolls)

(  result            rolled_at
 0      R  2026-03-02 12:01:40
 1      x  2026-03-02 12:01:44
 2      R  2026-03-02 12:01:48
 3      x  2026-03-02 12:01:51
 4     oo  2026-03-02 12:01:55,
 11)

In [4]:
if rolls.empty:
    raise ValueError('No rolls found in database. Record some rolls first.')

n = len(rolls)
observed_counts = rolls['result'].value_counts().to_dict()
rows = []

for symbol, p_expected in sorted(expected_probs.items()):
    k = observed_counts.get(symbol, 0)
    test = binomtest(k=k, n=n, p=p_expected, alternative='two-sided')
    rows.append({
        'symbol': symbol,
        'observed_count': k,
        'total_rolls': n,
        'observed_p': k / n,
        'expected_p': p_expected,
        'p_value': test.pvalue,
    })

results = pd.DataFrame(rows).sort_values('p_value')
results

,symbol,observed_count,total_rolls,observed_p,expected_p,p_value
0,R,3,11,0.272727,0.166667,0.407813
1,o,1,11,0.090909,0.166667,1.000000
2,oo,2,11,0.181818,0.166667,1.000000
3,x,3,11,0.272727,0.333333,1.000000
4,xx,2,11,0.181818,0.166667,1.000000


In [5]:
alpha = 0.05
results['reject_at_0_05'] = results['p_value'] < alpha
results

,symbol,observed_count,total_rolls,observed_p,expected_p,p_value,reject_at_0_05
0,R,3,11,0.272727,0.166667,0.407813,False
1,o,1,11,0.090909,0.166667,1.000000,False
2,oo,2,11,0.181818,0.166667,1.000000,False
3,x,3,11,0.272727,0.333333,1.000000,False
4,xx,2,11,0.181818,0.166667,1.000000,False
